In [1]:
import pandas as pd
import time
from openai import OpenAI

In [3]:
# Load CSV and subset
df = pd.read_csv("Y12-15_intake24_grouped_output.csv")
df = df[['ID', 'RecallNumber', 'MealName', 'SearchTerm', 'DescriptionEn']]

In [4]:
# Initialize OpenAI client

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# Function to format a batch prompt
def format_batch_prompt(batch_df):
    examples = []
    for i, row in enumerate(batch_df.itertuples(index=False)):
        example = f"{i+1}.\nSearchTerm: {row.SearchTerm}\nRecorded: {row.DescriptionEn}"
        examples.append(example)
    return "\n\n".join(examples)

# Parse model output back into list
def parse_model_output(output, expected_count):
    lines = output.strip().split("\n")
    results = [""] * expected_count
    for line in lines:
        if not line.strip():
            continue
        if line[0].isdigit():
            try:
                index, content = line.split(".", 1)
                idx = int(index.strip()) - 1
                if 0 <= idx < expected_count:
                    results[idx] = content.strip()
            except:
                continue
    return results

# Create a column for results
df['LostFoodsAnalysis'] = ""

# Process in batches of 10
batch_size = 10
# total_size = 1000
total_size = len(df)


# Prompt template
SYSTEM_PROMPT = (
    "You are an assistant that checks for missing food items in records of food made by study participants searching for the foods they have eaten.\n"
    "For each example, compare the foods searched for (SearchTerm) to the recorded foods (Recorded).\n"
    "List any foods from the search that are missing from the recorded list. If nothing is missing, leave the output blank.\n"
    "Note that the search terms might have spelling mistakes. A search for 'watee' likely matches a recorded item of 'water'.\n"
    "As an LLM you will also understand semantic matches: A search for 'chicken curry' would match 'chicken madras'.\n"
    "You can also understand approximate matches like 'cheese' and 'Cheshire cheese'.\n"
    "If you think you have a found a missing item, check it again carefully against the recorded food to make sure it is actually missing before returning it."
    "Return your results in the format:\n1. [missing foods or blank]\n2. [missing foods or blank]\n... up to {batch_size}."
)



for start in range(0, total_size, batch_size):
    end = start + batch_size
    batch_df = df.iloc[start:end].copy()
    if batch_df.empty:
        continue
    
    print(f"Processing rows {start+1} to {min(end, total_size)} of {total_size}...")
    
    prompt = format_batch_prompt(batch_df)

    try:
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
        )
        output = response.choices[0].message.content.strip()
        parsed_results = parse_model_output(output, len(batch_df))


        # Assign results back to the main DataFrame
        df.loc[start:end-1, 'LostFoodsAnalysis'] = parsed_results

    except Exception as e:
        df.loc[start:end-1, 'LostFoodsAnalysis'] = f"Error: {e}"

    time.sleep(1)  # avoid rate limits

# Save results
df.to_csv("lost_foods_output_batch.csv", index=False)

Processing rows 1 to 10 of 5534...
Processing rows 11 to 20 of 5534...
Processing rows 21 to 30 of 5534...
Processing rows 31 to 40 of 5534...
Processing rows 41 to 50 of 5534...
Processing rows 51 to 60 of 5534...
Processing rows 61 to 70 of 5534...
Processing rows 71 to 80 of 5534...
Processing rows 81 to 90 of 5534...
Processing rows 91 to 100 of 5534...
Processing rows 101 to 110 of 5534...
Processing rows 111 to 120 of 5534...
Processing rows 121 to 130 of 5534...
Processing rows 131 to 140 of 5534...
Processing rows 141 to 150 of 5534...
Processing rows 151 to 160 of 5534...
Processing rows 161 to 170 of 5534...
Processing rows 171 to 180 of 5534...
Processing rows 181 to 190 of 5534...
Processing rows 191 to 200 of 5534...
Processing rows 201 to 210 of 5534...
Processing rows 211 to 220 of 5534...
Processing rows 221 to 230 of 5534...
Processing rows 231 to 240 of 5534...
Processing rows 241 to 250 of 5534...
Processing rows 251 to 260 of 5534...
Processing rows 261 to 270 of 55

Processing rows 2111 to 2120 of 5534...
Processing rows 2121 to 2130 of 5534...
Processing rows 2131 to 2140 of 5534...
Processing rows 2141 to 2150 of 5534...
Processing rows 2151 to 2160 of 5534...
Processing rows 2161 to 2170 of 5534...
Processing rows 2171 to 2180 of 5534...
Processing rows 2181 to 2190 of 5534...
Processing rows 2191 to 2200 of 5534...
Processing rows 2201 to 2210 of 5534...
Processing rows 2211 to 2220 of 5534...
Processing rows 2221 to 2230 of 5534...
Processing rows 2231 to 2240 of 5534...
Processing rows 2241 to 2250 of 5534...
Processing rows 2251 to 2260 of 5534...
Processing rows 2261 to 2270 of 5534...
Processing rows 2271 to 2280 of 5534...
Processing rows 2281 to 2290 of 5534...
Processing rows 2291 to 2300 of 5534...
Processing rows 2301 to 2310 of 5534...
Processing rows 2311 to 2320 of 5534...
Processing rows 2321 to 2330 of 5534...
Processing rows 2331 to 2340 of 5534...
Processing rows 2341 to 2350 of 5534...
Processing rows 2351 to 2360 of 5534...


Processing rows 4161 to 4170 of 5534...
Processing rows 4171 to 4180 of 5534...
Processing rows 4181 to 4190 of 5534...
Processing rows 4191 to 4200 of 5534...
Processing rows 4201 to 4210 of 5534...
Processing rows 4211 to 4220 of 5534...
Processing rows 4221 to 4230 of 5534...
Processing rows 4231 to 4240 of 5534...
Processing rows 4241 to 4250 of 5534...
Processing rows 4251 to 4260 of 5534...
Processing rows 4261 to 4270 of 5534...
Processing rows 4271 to 4280 of 5534...
Processing rows 4281 to 4290 of 5534...
Processing rows 4291 to 4300 of 5534...
Processing rows 4301 to 4310 of 5534...
Processing rows 4311 to 4320 of 5534...
Processing rows 4321 to 4330 of 5534...
Processing rows 4331 to 4340 of 5534...
Processing rows 4341 to 4350 of 5534...
Processing rows 4351 to 4360 of 5534...
Processing rows 4361 to 4370 of 5534...
Processing rows 4371 to 4380 of 5534...
Processing rows 4381 to 4390 of 5534...
Processing rows 4391 to 4400 of 5534...
Processing rows 4401 to 4410 of 5534...


In [25]:
temp_df = df.head(10)
prompt = format_batch_prompt(batch_df)


In [27]:
SYSTEM_PROMPT

"You are an assistant that checks for missing food items.\nFor each example, compare the foods searched for (SearchTerm) to the recorded foods (DescriptionEn).\nList any foods from the search that are missing from the recorded list. If nothing is missing, leave the output blank.\nNote that the search terms might have spelling mistakes. So a search for 'watee' likely matches a recorded item of 'water'Return your results in the format:\n1. [missing foods or blank]\n2. [missing foods or blank]\n... up to 10."

In [26]:
prompt

"1.\nSearchTerm: oats\nRecorded: Oats (uncooked),Whole milk,Treacle\n\n2.\nSearchTerm: cauliflower,potatoes,chicken,brandy butter,custard,cocoa\nRecorded: Cauliflower,Potatoes,Butter, unsalted,Chicken slice with stuffing,Brandy butter,Custard, made with sugar (homemade),Cocoa powder,Sweet mince pie (filled with dried fruit)\n\n3.\nSearchTerm: shortbread,water\nRecorded: Shortbread,Water (from tap, including filtered)\n\n4.\nSearchTerm: peas,cheese,tomato,fruit cake,biscuit,fish pie,bread,tea\nRecorded: Peas,Cheddar cheese,Cherry tomatoes,Fruit cake, with icing  (with or without marzipan) e.g. Christmas cake,Chocolate coated biscuit (e.g. Rocky bar, Club, Breakaway),Fisherman's pie, home made,White bread,Butter, unsalted,Tea,Whole milk\n\n5.\nSearchTerm: Biscuit,tea\nRecorded: Shortcake biscuit,Tea,Whole milk\n\n6.\nSearchTerm: tea\nRecorded: Tea,Whole milk\n\n7.\nSearchTerm: bran flakes,stewed apple,yoghurt,tea\nRecorded: Bran Flakes, supermarket brands,Whole milk,Stewed apple,Natural 